# Model Analizi — Agnostic Framework

Herhangi bir HuggingFace CausalLM modelini yükle, mimarisini, parametrelerini ve weight'lerini incele.

### Kullanım
`MODEL_CONFIGS` dict'ine istediğin modeli ekle — tüm analizler otomatik çalışır.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"Device: {device}")

In [ ]:
# ============================================================
# Model Config — istediğin modeli ekle
# ============================================================

MODEL_CONFIGS = {
    "Kumru-2B": {"id": "vngrs-ai/Kumru-2B", "kwargs": {"torch_dtype": "auto", "device_map": "auto"}},
    #"Kara-Kumru": {"id": "AlicanKiraz0/Kara-Kumru-v1.0-2B", "kwargs": {"torch_dtype": "auto", "device_map": "auto"}},
    #"Turkish-GPT2": {"id": "ytu-ce-cosmos/turkish-gpt2", "kwargs": {}},
    #"Qwen3.5-0.8B": {"id": "Qwen/Qwen3.5-0.8B", "kwargs": {"torch_dtype": torch.bfloat16}},
}

MODELS = {}
TOKENIZERS = {}
CONFIGS = {}

for name, cfg in MODEL_CONFIGS.items():
    print(f"Yükleniyor: {name} ({cfg['id']})...")
    CONFIGS[name] = AutoConfig.from_pretrained(cfg["id"], trust_remote_code=True)
    TOKENIZERS[name] = AutoTokenizer.from_pretrained(cfg["id"], trust_remote_code=True)
    m = AutoModelForCausalLM.from_pretrained(cfg["id"], trust_remote_code=True, **cfg["kwargs"])
    MODELS[name] = m.to(device) if "device_map" not in cfg["kwargs"] else m
    MODELS[name].eval()
    clear_gpu()

print(f"\n{len(MODELS)} model yüklendi: {list(MODELS.keys())}")

In [ ]:
config.text_config

In [ ]:
text_cfg = config.text_config if hasattr(config, 'text_config') else config

summary = {
    "Özellik": [
        "Model Tipi", "Vocab Size", "Hidden Size", "Num Layers",
        "Num Attention Heads", "Num KV Heads", "Head Dim",
        "Intermediate Size (FFN)", "FFN Genişleme Oranı",
        "Max Context Length", "Activation", "Normalization",
        "Positional Encoding", "Attention Bias", "Weight Tying",
    ],
    "Değer": [
        config.model_type, text_cfg.vocab_size, text_cfg.hidden_size,
        text_cfg.num_hidden_layers, text_cfg.num_attention_heads,
        text_cfg.num_key_value_heads, text_cfg.head_dim,
        text_cfg.intermediate_size,
        f"{text_cfg.intermediate_size / text_cfg.hidden_size:.1f}x",
        text_cfg.max_position_embeddings, text_cfg.hidden_act,
        f"RMSNorm (eps={text_cfg.rms_norm_eps})", "RoPE",
        str(getattr(text_cfg, 'attention_bias', False)),
        str(config.tie_word_embeddings),
    ],
}

pd.DataFrame(summary).set_index("Özellik")



## 2. Model Yükleme

In [ ]:
# # Modelleri indir
# gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_NAME)
# gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_NAME)

qwen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True, dtype=torch.bfloat16)
qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

#print("Turkish GPT-2 yüklendi.")
print(f"{MODEL_NAME} yüklendi.")

## 3. Mimari Yapı İncelemesi

Her iki modelin katman yapısını görelim.

In [ ]:
# print("=" * 60)
# print("TURKISH GPT-2 MİMARİSİ")
# print("=" * 60)
# print(gpt2_model)

In [ ]:
print("=" * 60)
print(f"QWEN3.5-0.8B MİMARİSİ")
print("=" * 60)
print(qwen_model)

### Qwen3.5 Layer Types (Hybrid Mimari)

Qwen3.5, her 4 katmanda bir full attention, geri kalanlar linear attention kullanır:

In [ ]:
type(text_cfg)

In [ ]:
layer_types = text_cfg.layer_types
for i, lt in enumerate(layer_types):
    marker = "<<<" if lt == "full_attention" else ""
    print(f"  Layer {i:2d}: {lt} {marker}")

print(f"\nToplam: {len(layer_types)} katman")
print(f"  Full attention: {layer_types.count('full_attention')}")
print(f"  Linear attention: {layer_types.count('linear_attention')}")

## 4. Parametre Sayıları ve Dağılımları

In [ ]:
def param_summary(model, model_name):
    """Model parametrelerinin katman bazlı özeti"""
    rows = []
    total = 0
    for name, param in model.named_parameters():
        n = param.numel()
        total += n
        rows.append({
            "Parametre": name,
            "Shape": str(list(param.shape)),
            "Dtype": str(param.dtype),
            "Eleman Sayısı": n,
        })
    
    df = pd.DataFrame(rows)
    print(f"\n{'=' * 60}")
    print(f"{model_name}")
    print(f"{'=' * 60}")
    print(f"Toplam parametre: {total:,}")
    print(f"Unique parametre sayısı: {sum(p.numel() for p in set(model.parameters())):,}")
    print(f"Toplam katman (named_parameters): {len(rows)}")
    
    # Bellek kullanımı
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    print(f"Bellek (model weights): {total_bytes / 1024**3:.2f} GB")
    
    return df

# df_gpt2_params = param_summary(gpt2_model, "Turkish GPT-2")
df_qwen_params = param_summary(qwen_model, "Qwen3.5-0.8B")

In [ ]:
# Parametre dağılımı: hangi bileşen ne kadar yer kaplıyor?
def param_distribution(model, model_name):
    """Parametre gruplarına göre dağılım"""
    groups = {}
    for name, param in model.named_parameters():
        # Grup ismi çıkar: embedding, attention, ffn, norm, head
        if "emb" in name or "wte" in name or "wpe" in name or "embed" in name:
            group = "Embedding"
        elif "attn" in name or "self_attn" in name or "attention" in name:
            group = "Attention"
        elif "mlp" in name or "ffn" in name or "feed" in name:
            group = "FFN (MLP)"
        elif "norm" in name or "ln" in name:
            group = "Normalization"
        elif "head" in name or "lm_head" in name:
            group = "Output Head"
        else:
            group = "Diğer"
        
        groups[group] = groups.get(group, 0) + param.numel()
    
    total = sum(groups.values())
    print(f"\n{model_name} - Parametre Dağılımı:")
    print("-" * 50)
    for group, count in sorted(groups.items(), key=lambda x: -x[1]):
        pct = count / total * 100
        bar = "#" * int(pct / 2)
        print(f"  {group:<20s}: {count:>12,} ({pct:5.1f}%) {bar}")
    print(f"  {'TOPLAM':<20s}: {total:>12,}")

param_distribution(gpt2_model, "Turkish GPT-2")
param_distribution(qwen_model, "Qwen3.5-0.8B")

## 5. Weight İstatistikleri

Her katmanın weight'lerinin mean, std, min, max değerlerini inceleyelim. Bu bize modelin eğitim sonrası weight dağılımı hakkında bilgi verir.

In [ ]:
def weight_statistics(model, model_name, top_n=20):
    """Weight istatistikleri tablosu"""
    rows = []
    for name, param in model.named_parameters():
        data = param.detach().float()
        rows.append({
            "Parametre": name,
            "Shape": str(list(param.shape)),
            "Mean": f"{data.mean().item():.6f}",
            "Std": f"{data.std().item():.6f}",
            "Min": f"{data.min().item():.6f}",
            "Max": f"{data.max().item():.6f}",
            "Abs Mean": f"{data.abs().mean().item():.6f}",
        })
    
    df = pd.DataFrame(rows)
    print(f"\n{model_name} - Weight İstatistikleri (ilk {top_n} katman):")
    return df

df_gpt2_stats = weight_statistics(gpt2_model, "Turkish GPT-2")
df_gpt2_stats.head(20)

In [ ]:
df_qwen_stats = weight_statistics(qwen_model, "Qwen3.5-0.8B")
df_qwen_stats.head(20)

### Weight Dağılımı Görselleştirme

İlk transformer bloğunun attention ve FFN weight'lerinin histogramları:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# GPT-2: İlk blok attention + MLP
gpt2_attn_w = gpt2_model.transformer.h[0].attn.c_attn.weight.detach().float().flatten().numpy()
gpt2_mlp_w = gpt2_model.transformer.h[0].mlp.c_fc.weight.detach().float().flatten().numpy()

axes[0, 0].hist(gpt2_attn_w, bins=100, alpha=0.7, color="steelblue")
axes[0, 0].set_title("GPT-2 - Layer 0 Attention Weight")
axes[0, 0].set_xlabel("Weight değeri")

axes[0, 1].hist(gpt2_mlp_w, bins=100, alpha=0.7, color="coral")
axes[0, 1].set_title("GPT-2 - Layer 0 MLP Weight")
axes[0, 1].set_xlabel("Weight değeri")

# Qwen: Layer 3 full_attention (self_attn var)
qwen_full_attn_layer = qwen_model.model.layers[3]
qwen_attn_w = qwen_full_attn_layer.self_attn.q_proj.weight.detach().float().flatten().numpy()
qwen_mlp_w = qwen_full_attn_layer.mlp.gate_proj.weight.detach().float().flatten().numpy()

axes[1, 0].hist(qwen_attn_w, bins=100, alpha=0.7, color="steelblue")
axes[1, 0].set_title("Qwen3.5 - Layer 3 (full_attn) q_proj Weight")
axes[1, 0].set_xlabel("Weight değeri")

axes[1, 1].hist(qwen_mlp_w, bins=100, alpha=0.7, color="coral")
axes[1, 1].set_title("Qwen3.5 - Layer 3 MLP gate_proj Weight")
axes[1, 1].set_xlabel("Weight değeri")

plt.tight_layout()
plt.show()

### Katman Bazlı Std Değişimi

Eğitilmiş bir modelde weight std'sinin katmanlar boyunca nasıl değiştiğini görelim. Bu, modelin derinlik boyunca bilgiyi nasıl işlediğini gösterir:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GPT-2: her blok için attention weight std
gpt2_stds = []
for i, block in enumerate(gpt2_model.transformer.h):
    std = block.attn.c_attn.weight.detach().float().std().item()
    gpt2_stds.append(std)

axes[0].plot(gpt2_stds, marker="o", color="steelblue")
axes[0].set_title("GPT-2 - Attention Weight Std (katman bazlı)")
axes[0].set_xlabel("Katman")
axes[0].set_ylabel("Std")
axes[0].grid(True, alpha=0.3)

# Qwen: her blok için MLP gate_proj std (tüm katmanlarda mevcut)
qwen_stds = []
qwen_labels = []
for i, layer in enumerate(qwen_model.model.layers):
    std = layer.mlp.gate_proj.weight.detach().float().std().item()
    qwen_stds.append(std)
    qwen_labels.append(layer.layer_type[:3])  # "lin" veya "ful"

colors = ["coral" if lt == "lin" else "steelblue" for lt in qwen_labels]
axes[1].bar(range(len(qwen_stds)), qwen_stds, color=colors)
axes[1].set_title("Qwen3.5 - MLP gate_proj Std (turuncu=linear, mavi=full_attn)")
axes[1].set_xlabel("Katman")
axes[1].set_ylabel("Std")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Inference Denemesi

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### Turkish GPT-2 ile metin üretimi

In [ ]:
from transformers import pipeline

# Turkish GPT-2
gpt2_model.to(device)
gpt2_generator = pipeline(
    "text-generation",
    model=gpt2_model,
    tokenizer=gpt2_tokenizer,
    device=device,
)

prompt_tr = "Yapay zekanın geleceği"
result = gpt2_generator(prompt_tr, max_length=100, do_sample=True, temperature=0.7, top_k=50)
print(f"Prompt: {prompt_tr}")
print(f"Çıktı:\n{result[0]['generated_text']}")

### Qwen3.5 ile metin üretimi

In [ ]:
qwen_model.to(device)

prompt_en = "Yapay zekanın geleceği"
inputs = qwen_tokenizer(prompt_en, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_k=50,
    )

generated = qwen_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt_en}")
print(f"Çıktı:\n{generated}")

## 7. Özet

| | Turkish GPT-2 | Qwen3.5-0.8B |
|---|---|---|
| **Parametre** | ~124M | ~800M |
| **Mimari** | Klasik GPT-2 (MHA + GELU + LayerNorm) | Hybrid (Linear Attn + Full Attn + SwiGLU + RMSNorm) |
| **Attention** | Multi-Head (12 head, tümü KV) | GQA (8 head, 2 KV) + Gated DeltaNet |
| **Pozisyon** | Learned embeddings (1024 max) | RoPE (262K context) |
| **FFN** | 4x genişleme, GELU | 3.5x genişleme, SwiGLU (gated) |
| **Dil** | Türkçe | Çok dilli |
| **Weight tying** | Var | Var |

**Temel gözlemler:**
- GPT-2 klasik bir decoder-only transformer — basit, anlaşılır yapı
- Qwen3.5 hybrid mimari ile linear attention (DeltaNet) ve full attention'ı birleştiriyor
- GQA ile KV cache bellek kullanımı 4x azalıyor (8 head vs 2 KV head)
- SwiGLU activation, GELU'dan daha iyi performans veriyor (ek bir gate projection ile)
- RoPE, learned positional embedding'e göre çok daha uzun context destekliyor

---
---
# Part 2: Turkish GPT-2 (ytu-ce-cosmos/turkish-gpt2)

Klasik GPT-2 mimarisi. Part 1'deki Qwen3.5 ile karşılaştırarak "eski vs yeni" farkını göreceğiz.

**Bileşenler:**
- `LayerNorm` (mean + var + scale + shift)
- `GELU` activation
- `FeedForward` (Linear → GELU → Linear, 4x genişleme)
- `MultiHeadAttention` (tüm head'ler kendi Q,K,V — birleşik c_attn matrisi)
- Learned Positional Embedding (wpe)

## P2.1 Config ve Model Yükleme

In [ ]:
from transformers import GPT2LMHeadModel,AutoConfig,AutoTokenizer
from transformers import pipeline

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
GPT2_NAME = "ytu-ce-cosmos/turkish-gpt2"
gpt2_config = AutoConfig.from_pretrained(GPT2_NAME)
gpt2_model = GPT2LMHeadModel.from_pretrained(GPT2_NAME)
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_NAME)
gpt2_model.eval()

print(gpt2_config)

In [ ]:
text_generator = pipeline('text-generation', model=gpt2_model, tokenizer=gpt2_tokenizer)
r = text_generator("Teknolojinin gelişimi hayatımızı önemli ölçüde etkiledi. ", max_length=100)
[{'generated_text': 'Teknolojinin gelişimi hayatımızı önemli ölçüde etkiledi. "Dijitalleşme" ile birlikte hayatımızın belirli bir parçası daha rahata ermeye başladı.'}]

In [ ]:
gpt2_summary = {
    "Özellik": [
        "Model Tipi", "Vocab Size", "Hidden Size (n_embd)", "Num Layers",
        "Num Attention Heads", "Head Dim", "Intermediate Size (FFN)",
        "FFN Genişleme", "Max Position", "Activation",
        "Normalization", "Positional Encoding", "Bias", "Weight Tying",
    ],
    "Değer": [
        gpt2_config.model_type, gpt2_config.vocab_size, gpt2_config.n_embd,
        gpt2_config.n_layer, gpt2_config.n_head,
        gpt2_config.n_embd // gpt2_config.n_head,
        gpt2_config.n_inner if gpt2_config.n_inner else gpt2_config.n_embd * 4,
        f"{(gpt2_config.n_inner or gpt2_config.n_embd * 4) / gpt2_config.n_embd:.0f}x",
        gpt2_config.n_positions, gpt2_config.activation_function,
        "LayerNorm", "Learned Embedding", "Var",
        str(getattr(gpt2_config, 'tie_word_embeddings', True)),
    ],
}
pd.DataFrame(gpt2_summary).set_index("Özellik")

## P2.2 Mimari Yapı

In [ ]:
print(gpt2_model)

## P2.3 Bileşen Detayları

### Embedding: Token + Position (Learned)

In [ ]:
wte = gpt2_model.transformer.wte
wpe = gpt2_model.transformer.wpe
lm_head = gpt2_model.lm_head

print(f"Token Embedding (wte): {wte}")
print(f"  vocab_size: {wte.num_embeddings:,}, emb_dim: {wte.embedding_dim}")
print(f"  bellek: {wte.weight.numel() * wte.weight.element_size() / 1024**2:.1f} MB")
print(f"\nPosition Embedding (wpe): {wpe}")
print(f"  max_pos: {wpe.num_embeddings}, emb_dim: {wpe.embedding_dim}")
print(f"  bellek: {wpe.weight.numel() * wpe.weight.element_size() / 1024**2:.1f} MB")
print(f"\nlm_head: {lm_head}")
print(f"Weight tying (wte == lm_head): {wte.weight.data_ptr() == lm_head.weight.data_ptr()}")
print(f"\nQwen'de: sadece embed_tokens (pos_emb yok, RoPE attention içinde), weight tying var")

### LayerNorm

In [ ]:
ln = gpt2_model.transformer.h[0].ln_1
print(f"LayerNorm: {ln}")
print(f"  weight (scale): {list(ln.weight.shape)}")
print(f"  bias (shift)  : {list(ln.bias.shape)}")
print(f"  eps: {ln.eps}")
print(f"  Parametre: {ln.weight.numel() + ln.bias.numel()} (scale + shift)")
print(f"\nQwen'de: RMSNorm — sadece scale ({ln.weight.numel()} param), bias yok")

### Multi-Head Attention

In [ ]:
gpt2_model.transformer

In [ ]:
attn = gpt2_model.transformer.h[0].attn
print(f"Attention: {type(attn).__name__}")
for name, mod in attn.named_children():
    shape = f" {list(mod.weight.shape)}" if hasattr(mod, 'weight') else ""
    print(f"  {name}: {type(mod).__name__}{shape}")

c_attn = attn.c_attn
n = gpt2_config.n_embd
print(f"\nc_attn: {list(c_attn.weight.shape)} — Q,K,V birleşik tek matris")
print(f"  Q: [:, 0:{n}],  K: [:, {n}:{n*2}],  V: [:, {n*2}:{n*3}]")
print(f"\nHeads: {gpt2_config.n_head}, Head dim: {n // gpt2_config.n_head}")
print(f"MHA: tüm {gpt2_config.n_head} head kendi Q,K,V'sine sahip")
print(f"\nQwen'de: GQA — {gpt2_config.n_head} query head, 2 KV head (shared)")

### FeedForward (MLP)

In [ ]:
mlp = gpt2_model.transformer.h[0].mlp
print(f"MLP:")
for name, mod in mlp.named_children():
    shape = f" {list(mod.weight.shape)}" if hasattr(mod, 'weight') else ""
    print(f"  {name}: {type(mod).__name__}{shape}")

inter = gpt2_config.n_inner if gpt2_config.n_inner else gpt2_config.n_embd * 4
print(f"\nYapı: {gpt2_config.n_embd} -> {inter} (c_fc) -> GELU -> {gpt2_config.n_embd} (c_proj)")
print(f"Genişleme: {inter / gpt2_config.n_embd:.0f}x, 2 matris")
print(f"\nQwen'de: SwiGLU — 3 matris (gate + up + down), 3.5x genişleme")

## P2.4 Parametre Sayıları ve Dağılımları

In [ ]:
total = sum(p.numel() for p in gpt2_model.parameters())
unique = sum(p.numel() for p in set(gpt2_model.parameters()))
total_bytes = sum(p.numel() * p.element_size() for p in gpt2_model.parameters())

print(f"Toplam parametre : {total:,}")
print(f"Unique parametre : {unique:,}")
print(f"Tied             : {total - unique:,}")
print(f"Bellek (float32) : {total_bytes / 1024**3:.2f} GB")

# Dağılım
groups = {}
for name, param in gpt2_model.named_parameters():
    if "wte" in name or "wpe" in name:
        group = "Embedding (wte+wpe)"
    elif "attn" in name:
        group = "Attention"
    elif "mlp" in name:
        group = "FFN (MLP)"
    elif "ln" in name:
        group = "LayerNorm"
    elif "lm_head" in name:
        group = "Output Head"
    else:
        group = "Diğer"
    groups[group] = groups.get(group, 0) + param.numel()

t = sum(groups.values())
print(f"\nParametre Dağılımı:")
print("-" * 55)
for group, count in sorted(groups.items(), key=lambda x: -x[1]):
    pct = count / t * 100
    bar = "#" * int(pct / 2)
    print(f"  {group:<25s}: {count:>12,} ({pct:5.1f}%) {bar}")

## P2.5 Weight İstatistikleri ve Görselleştirme

In [ ]:
df_gpt2_stats = weight_statistics(gpt2_model, "Turkish GPT-2")
df_gpt2_stats.head(15)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

weights = [
    ("Block 0 — Attention (c_attn)", gpt2_model.transformer.h[0].attn.c_attn.weight),
    ("Block 0 — MLP (c_fc)", gpt2_model.transformer.h[0].mlp.c_fc.weight),
    (f"Block {gpt2_config.n_layer-1} — Attention", gpt2_model.transformer.h[-1].attn.c_attn.weight),
    (f"Block {gpt2_config.n_layer-1} — MLP", gpt2_model.transformer.h[-1].mlp.c_fc.weight),
]

for ax, (title, w) in zip(axes.flat, weights):
    data = w.detach().float().flatten().numpy()
    ax.hist(data, bins=100, alpha=0.7, color="steelblue")
    ax.set_title(f"{title}\nmean={data.mean():.4f}, std={data.std():.4f}")
    ax.set_xlabel("Weight değeri")

plt.suptitle("GPT-2 Weight Histogramları", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Katman bazlı std
attn_stds = [b.attn.c_attn.weight.detach().float().std().item() for b in gpt2_model.transformer.h]
mlp_stds = [b.mlp.c_fc.weight.detach().float().std().item() for b in gpt2_model.transformer.h]

fig, ax = plt.subplots(figsize=(10, 4))
x = range(gpt2_config.n_layer)
ax.plot(x, attn_stds, marker="o", label="Attention (c_attn)", color="steelblue")
ax.plot(x, mlp_stds, marker="s", label="MLP (c_fc)", color="coral")
ax.set_xlabel("Katman"); ax.set_ylabel("Weight Std")
ax.set_title("GPT-2 — Katman bazlı weight std")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## P2.6 Forward Pass ve Inference

In [ ]:
gpt2_model.to(device)

text = "Yapay zekanın geleceği"
inputs = gpt2_tokenizer(text, return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

print(f"Input: '{text}'")
print(f"Token IDs: {input_ids.tolist()}")
for tid in input_ids[0]:
    print(f"  {tid.item():>6d} -> '{gpt2_tokenizer.decode([tid.item()])}'")

with torch.no_grad():
    # Adım adım
    tok = gpt2_model.transformer.wte(input_ids)
    pos = gpt2_model.transformer.wpe(torch.arange(input_ids.shape[1], device=device))
    hidden = tok + pos
    print(f"\ntok_emb: {list(tok.shape)}")
    print(f"pos_emb: {list(pos.shape)}")
    print(f"hidden (tok+pos): {list(hidden.shape)}")

    # Full forward
    logits = gpt2_model(input_ids).logits
    print(f"\nLogits: {list(logits.shape)}")

    probs = torch.softmax(logits[0, -1].float(), dim=-1)
    top5 = torch.topk(probs, 5)
    print(f"\nNext token (top 5):")
    for p, idx in zip(top5.values, top5.indices):
        print(f"  '{gpt2_tokenizer.decode([idx.item()])}' prob={p.item():.4f}")

In [ ]:
# Metin üretimi
prompts = ["Yapay zekanın geleceği", "Türkiye'nin en büyük", "Teknolojinin gelişimi hayatımızı"]

for prompt in prompts:
    inp = gpt2_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = gpt2_model.generate(
            **inp, max_new_tokens=50, do_sample=True,
            temperature=0.7, top_k=50, pad_token_id=gpt2_tokenizer.eos_token_id,
        )
    print(f"Prompt: '{prompt}'")
    print(f"  {gpt2_tokenizer.decode(out[0], skip_special_tokens=True)[:200]}\n")